# 08. Motion Attribution Test

This notebook tests the motion attribution module on annotated, normalized pose data.

Motion attribution runs after normalization and before feature extraction:

```
Validation → Annotation → Exercise Definition → Preprocessing → Normalization → Motion Attribution → Features
```

The module verifies, at the repetition level, whether the limb that actually moved in each rep
matches the limb that should have moved according to the declared exercise pattern.

This module is **exercise-aware**:

```text
pattern = bilateral     module is skipped — no per-rep active limb concept
pattern = alternating   module runs and produces attribution metadata
pattern unknown         module is skipped (safe default)
```

Output columns added by motion attribution (populated only on `rep` frames):

- `detected_active_limb` — `'left'` | `'right'` | `'bilateral'` | `'ambiguous'` | `None`
- `expected_active_limb` — `'left'` | `'right'` | `None`
- `attribution_consistent` — `bool` | `None`
- `attribution_confidence` — `float` (0 to 1) | `None`
- `attribution_action` — `'accept'` | `'flag'` | `'swap'` | `None`

This notebook assumes that the previous checks are already working:

- 00_environment_check
- 01_data_loading_test
- 02_validation_test
- 03_raw_visualization_test
- 04_normalization_test
- 05_annotation_mask_test
- 07_preprocessing_test

> **Status:** The `motion_attribution` module is not yet implemented.
> Running the pipeline with `motion_attribution.enabled: true` raises `NotImplementedError`.
> This notebook documents the design and will be updated when the module is ready.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json

import pandas as pd

from movement.annotation import apply_annotation, load_annotation_csv
from movement.config import LANDMARKS, make_coordinate_columns, make_required_columns, make_visibility_columns
from movement.io import load_pose_csv
from movement.normalization import normalize_pose_by_hip_torso
from movement.pipeline import MotionAttributionConfig, PipelineConfig, run_pipeline
from movement.validation import run_basic_validation

## Data Setup

Load pose CSV, validate, annotate, and normalize.
Motion attribution requires normalized coordinates and rep boundaries from annotation.

In [ ]:
csv_path = "../data/sample/mediapipe_squat_synthetic.csv"
ann_path = "../data/sample/mediapipe_squat_synthetic_annotation.csv"

df = load_pose_csv(csv_path)

val_report = run_basic_validation(
    df=df,
    required_columns=make_required_columns(LANDMARKS),
    coordinate_columns=make_coordinate_columns(LANDMARKS),
    visibility_columns=make_visibility_columns(LANDMARKS),
)
print("validation passed:", val_report["passed"])

In [ ]:
ann_df = load_annotation_csv(ann_path)
annotated_df, ann_report = apply_annotation(df, ann_df)

norm_df, norm_report = normalize_pose_by_hip_torso(df=annotated_df, landmarks=LANDMARKS)

print(f"normalized dataframe: {norm_df.shape[0]} frames, {norm_df.shape[1]} columns")

## Module Scope: Bilateral Exercise → Skipped

The sample data is annotated as `exercise_type='forward_bend'`, `pattern='bilateral'`.

For bilateral exercises there is no expected per-rep active side,
so motion attribution is not applicable and the module will be skipped.

In [ ]:
print("exercise context in normalized dataframe:")
print("  exercise_type:", norm_df["exercise_type"].dropna().unique().tolist())
print("  pattern:      ", norm_df["pattern"].dropna().unique().tolist())
print()
print("rep segments in annotation:")
rep_rows = norm_df[norm_df["segment_type"] == "rep"][
    ["frame", "set_id", "rep_id", "segment_type"]
].groupby(["set_id", "rep_id"]).agg(
    start=("frame", "min"), end=("frame", "max"), count=("frame", "count")
)
print(rep_rows.to_string())

## Current Status: Module Not Implemented

The pipeline raises `NotImplementedError` when `motion_attribution.enabled` is `True`.

This is the expected behaviour during development.

In [ ]:
config_not_implemented = PipelineConfig()
config_not_implemented.motion_attribution = MotionAttributionConfig(enabled=True)
config_not_implemented.validation.enabled = False
config_not_implemented.normalization.enabled = False
config_not_implemented.annotation.enabled = False

try:
    run_pipeline(norm_df, config=config_not_implemented)
except NotImplementedError as e:
    print("NotImplementedError raised as expected:")
    print(" ", e)

## Active Limb Detection Logic (Design Reference)

For each rep window, motion energy is computed for left and right paired landmarks.

```text
left_motion  = Σ |p_left(t+1)  - p_left(t)|    for t in rep window
right_motion = Σ |p_right(t+1) - p_right(t)|   for t in rep window

motion_share = max(left_motion, right_motion)
             / (left_motion + right_motion + ε)
```

Landmark pairs used per exercise:

```text
plank_shoulder_tap  :  left_wrist  vs  right_wrist
lunge               :  left_knee   vs  right_knee
                       left_ankle  vs  right_ankle
```

Detection thresholds (defaults):

```text
τ_active    = 0.70    motion_share > 0.70  → clear active side detected
τ_ambiguous = 0.55    0.55 < motion_share ≤ 0.70  → ambiguous
τ_swap      = 0.85    swap correction only when confidence > 0.85
```

## Example: Alternating Exercise Annotation

For an alternating exercise such as `plank_shoulder_tap`, the expected active limb
alternates between reps based on `starting_side`.

Example annotation (not the current sample data):

```csv
segment_type,set_id,rep_id,start_frame,end_frame,use_for_analysis,exercise_type,pattern,starting_side
baseline,,,0,40,false,plank_shoulder_tap,alternating,right
rep,1,1,50,100,true,plank_shoulder_tap,alternating,right
rep,1,2,110,160,true,plank_shoulder_tap,alternating,right
rep,1,3,170,220,true,plank_shoulder_tap,alternating,right
rep,1,4,230,280,true,plank_shoulder_tap,alternating,right
```

With `starting_side=right`, the expected active limb per rep would be:

```text
rep 1  →  right
rep 2  →  left
rep 3  →  right
rep 4  →  left
```

Motion attribution would compare the detected active limb against this expected sequence
and produce `attribution_consistent` and `attribution_action` per rep.

## Interpretation

Once the module is implemented, expected checks are:

**For bilateral exercises (e.g. the current sample data):**

- module is skipped automatically
- `detected_active_limb`, `expected_active_limb`, `attribution_consistent` remain `None`
- attribution report records `pattern='bilateral'` and `num_reps=0` (skipped)

**For alternating exercises (e.g. lunge, plank_shoulder_tap):**

- `detected_active_limb` is `'left'` or `'right'` when motion_share > τ_active
- `expected_active_limb` matches the declared `starting_side` alternation pattern
- `attribution_consistent` is `True` when detected matches expected
- `attribution_action` is `'flag'` in conservative mode (default) when inconsistency is found
- original coordinate values are never modified
- original `frame` and `rep_id` boundaries are never modified